# Experiment - ResNet18 on tiles

The pivot to multiple-instance learning: each slide becomes a bag of 256x256 tiles, every tile is classified independently, and the slide prediction is the mean of its tiles' softmax probabilities.

**Test F1: 0.3859** with flip test-time augmentation. A smaller backbone than the ResNet50 baseline scores higher here, because each tile is now seen at native resolution.

---


In [ ]:
# --- Paths ---------------------------------------------------------------
# Originally executed on Google Colab with the dataset on Google Drive.
# DATA_DIR must contain train_data/, test_data/ and train_labels.csv
# (see data/README.md).
import os

DATA_DIR = os.environ.get("WSI_DATA_DIR", "data")
os.makedirs("models", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)


In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd

In [3]:
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["tiles"], d["labels"], d["slide_ids"]

train_tiles, train_labels, train_slide_ids = load_npz("train_tiles_raw.npz")
test_tiles, _, test_slide_ids = load_npz("test_tiles_raw_3.npz")

print(train_tiles.shape, train_labels.shape)
print(test_tiles.shape)

(3392, 256, 256, 3) (3392,)
(2547, 256, 256, 3)


In [5]:
class TileDataset(Dataset):
    def __init__(self, tiles, labels, transform=None):
        self.tiles = tiles
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        img = torch.from_numpy(self.tiles[i]).permute(2,0,1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        y = int(self.labels[i])
        return img, y

In [19]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [23]:
train_ds = TileDataset(train_tiles, train_labels, transform=train_tf)
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [24]:
model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes=4
).to(device)

In [25]:
class_counts = np.bincount(train_labels, minlength=4)
class_weights = (class_counts.sum() / class_counts).astype(np.float32)
class_weights = class_weights / class_weights.mean()
weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

In [26]:
EPOCHS = 15

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch}/{EPOCHS} - loss: {running_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), "resnet18_multiclass_secondtry_1212.pth")

Epoch 1/15 - loss: 1.4136
Epoch 2/15 - loss: 1.3824
Epoch 3/15 - loss: 1.3509
Epoch 4/15 - loss: 1.3143
Epoch 5/15 - loss: 1.2917
Epoch 6/15 - loss: 1.2616
Epoch 7/15 - loss: 1.2474
Epoch 8/15 - loss: 1.2241
Epoch 9/15 - loss: 1.2022
Epoch 10/15 - loss: 1.1743
Epoch 11/15 - loss: 1.1534
Epoch 12/15 - loss: 1.1255
Epoch 13/15 - loss: 1.1046
Epoch 14/15 - loss: 1.0859
Epoch 15/15 - loss: 1.0626


In [27]:
torch.save(model.state_dict(), "resnet18_multiclass_secondtry_1212.pth")

In [28]:
import torchvision.transforms.functional as TF

@torch.no_grad()
def predict_slide_tta_meanprob(model, tiles, idxs, transform):
    imgs = torch.from_numpy(tiles[idxs]).permute(0,3,1,2).float() / 255.0
    imgs = torch.stack([transform(im) for im in imgs], dim=0)

    model.eval()
    bs = 64
    all_probs = []

    for i in range(0, imgs.size(0), bs):
        batch = imgs[i:i+bs].to(device)

        p0 = torch.softmax(model(batch), dim=1)
        p1 = torch.softmax(model(TF.hflip(batch)), dim=1)
        p2 = torch.softmax(model(TF.vflip(batch)), dim=1)

        p = (p0 + p1 + p2) / 3.0
        all_probs.append(p.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    return all_probs.mean(dim=0)

In [29]:
from collections import defaultdict

def build_slide_index(slide_ids):
    d = defaultdict(list)
    for i, sid in enumerate(slide_ids):
        d[sid].append(i)
    return d

train_slide_map = build_slide_index(train_slide_ids)
test_slide_map  = build_slide_index(test_slide_ids)

In [30]:
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))

test_preds = []
for sid, idxs in test_slide_map.items():
    prob = predict_slide_tta_meanprob(model, test_tiles, idxs, val_tf)
    test_preds.append((sid, prob.argmax().item()))

In [32]:
label_map = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

submission = pd.DataFrame({
    "sample_index": [sid for sid, _ in test_preds],
    "label": [label_map[p] for _, p in test_preds]
})

submission.to_csv("submission_resnet18_multiclass_1212_secondtry.csv", index=False)
submission.head()

,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal B
2,img_0002.png,Luminal B
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [33]:
import pandas as pd
import re

# carica submission
sub = pd.read_csv("submission_resnet18_multiclass_1212_secondtry.csv")

print("Righe totali:", len(sub))
print("Colonne:", sub.columns.tolist())

# ---- 1) column check ----
assert list(sub.columns) == ["sample_index", "label"], \
    f"Colonne errate: {sub.columns.tolist()}"

# ---- 2) sample_index format check ----
pattern = re.compile(r"^img_\d{4}\.png$")
invalid_idx = sub[~sub["sample_index"].astype(str).str.match(pattern)]

assert len(invalid_idx) == 0, \
    f"invalid sample_index values:\n{invalid_idx.head()}"

# ---- 3) range check, img_0000.png to img_0476.png ----
# extract the numeric part
sub["num"] = sub["sample_index"].str.extract(r"img_(\d{4})\.png").astype(int)

expected = set(range(0, 477))   # 0000 → 0476
found = set(sub["num"].tolist())

missing = sorted(expected - found)
extra   = sorted(found - expected)

assert not missing, f"File mancanti: {missing[:10]}..."
assert not extra,   f"File extra: {extra[:10]}..."

# ---- 4) valid-label check ----
valid_labels = {
    "Luminal A",
    "Luminal B",
    "HER2(+)",
    "Triple negative"
}

invalid_labels = sub[~sub["label"].isin(valid_labels)]

assert len(invalid_labels) == 0, \
    f"invalid labels:\n{invalid_labels.head()}"

# ---- 5) riepilogo distribuzione ----
print("\nDistribuzione label:")
print(sub["label"].value_counts())

print("\n Submission valid: format, indices and labels are correct.")

Righe totali: 477
Colonne: ['sample_index', 'label']

Distribuzione label:
label
Luminal A          156
Luminal B          124
HER2(+)            108
Triple negative     89
Name: count, dtype: int64

 Submission valid: format, indices and labels are correct.


test set score: 0.3859

Inference con rotation TTA

In [39]:
import torch
import torchvision.transforms.functional as TF

@torch.no_grad()
def predict_slide_tta_rot(
    model,
    tiles,          # np.ndarray (N, H, W, 3)
    idxs,           # indices of tiles for this slide
    transform
):
    model.eval()

    imgs = torch.from_numpy(tiles[idxs]).permute(0, 3, 1, 2).float() / 255.0
    imgs = torch.stack([transform(im) for im in imgs], dim=0)

    bs = 64
    all_probs = []

    for i in range(0, imgs.size(0), bs):
        batch = imgs[i:i+bs].to(device)

        # rotation TTA (safe: used in training)
        p0 = torch.softmax(model(batch), dim=1)
        p1 = torch.softmax(model(TF.rotate(batch, 90)), dim=1)
        p2 = torch.softmax(model(TF.rotate(batch, 180)), dim=1)
        p3 = torch.softmax(model(TF.rotate(batch, 270)), dim=1)

        p = (p0 + p1 + p2 + p3) / 4.0
        all_probs.append(p.cpu())

    all_probs = torch.cat(all_probs, dim=0)  # (N_patches, 4)
    return all_probs.mean(dim=0)

In [40]:
from collections import defaultdict

def build_slide_index(slide_ids):
    d = defaultdict(list)
    for i, sid in enumerate(slide_ids):
        d[sid].append(i)
    return d

train_slide_map = build_slide_index(train_slide_ids)
test_slide_map  = build_slide_index(test_slide_ids)

In [41]:
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))

test_preds = []

for slide_id, idxs in test_slide_map.items():
    prob = predict_slide_tta_rot(
        model,
        test_tiles,
        idxs,
        val_tf
    )
    pred = prob.argmax().item()
    test_preds.append((slide_id, pred))

In [42]:
label_map = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

submission = pd.DataFrame({
    "sample_index": [sid for sid, _ in test_preds],
    "label": [label_map[p] for _, p in test_preds]
})

submission.to_csv("submission_resnet18_multiclass_1212_ttarot.csv", index=False)
submission.head()

,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal B
2,img_0002.png,Luminal B
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [43]:
import pandas as pd

CSV_OLD = "submission_resnet18_multiclass_1212_secondtry.csv"        # 0.3859
CSV_NEW = "submission_resnet18_multiclass_1212_ttarot.csv"  # step 2

# Load
df_old = pd.read_csv(CSV_OLD)
df_new = pd.read_csv(CSV_NEW)

# Safety checks
assert set(df_old.columns) == {"sample_index", "label"}
assert set(df_new.columns) == {"sample_index", "label"}

df_old = df_old.sort_values("sample_index").reset_index(drop=True)
df_new = df_new.sort_values("sample_index").reset_index(drop=True)

assert (df_old["sample_index"] == df_new["sample_index"]).all(), \
    "Mismatch in sample_index ordering"

# Diff
diff = df_old.copy()
diff["label_old"] = df_old["label"]
diff["label_new"] = df_new["label"]
diff["changed"] = diff["label_old"] != diff["label_new"]

n_total = len(diff)
n_changed = diff["changed"].sum()

print(f"Total samples: {n_total}")
print(f"Changed predictions: {n_changed} ({100*n_changed/n_total:.2f}%)")

# Show distribution of changes
if n_changed > 0:
    print("\nChange breakdown (old → new):")
    print(
        diff[diff["changed"]]
        .groupby(["label_old", "label_new"])
        .size()
        .sort_values(ascending=False)
    )

# Optional: show first N changed samples
N_SHOW = 20
if n_changed > 0:
    print(f"\nFirst {min(N_SHOW, n_changed)} changed samples:")
    display(
        diff[diff["changed"]]
        .head(N_SHOW)[["sample_index", "label_old", "label_new"]]
    )
else:
    print("\nNo differences found between submissions.")

Total samples: 477
Changed predictions: 42 (8.81%)

Change breakdown (old → new):
label_old        label_new
Luminal B        Triple negative    6
                 Luminal A          5
HER2(+)          Luminal B          4
Luminal A        HER2(+)            4
Triple negative  Luminal B          4
                 HER2(+)            4
Luminal B        HER2(+)            4
Luminal A        Luminal B          4
HER2(+)          Triple negative    3
                 Luminal A          2
Luminal A        Triple negative    1
Triple negative  Luminal A          1
dtype: int64

First 20 changed samples:


,sample_index,label_old,label_new
5,img_0005.png,Luminal B,Triple negative
9,img_0009.png,Luminal A,HER2(+)
13,img_0013.png,Luminal B,Triple negative
52,img_0052.png,Luminal B,Luminal A
61,img_0061.png,HER2(+),Luminal A
65,img_0065.png,Luminal A,HER2(+)
68,img_0068.png,HER2(+),Luminal B
110,img_0110.png,Luminal B,HER2(+)
122,img_0122.png,Luminal B,HER2(+)
128,img_0128.png,Luminal A,HER2(+)


Same result

TTA combining (Flip + rot)

In [44]:
import torch
import torchvision.transforms.functional as TF

@torch.no_grad()
def predict_slide_tta_dihedral(model, tiles, idxs, transform):
    model.eval()

    # Prepare the base image batch
    imgs = torch.from_numpy(tiles[idxs]).permute(0, 3, 1, 2).float() / 255.0
    imgs = torch.stack([transform(im) for im in imgs], dim=0)

    bs = 64
    all_probs = []

    for i in range(0, imgs.size(0), bs):
        batch = imgs[i:i+bs].to(device) # Batch originale
        batch_flip = TF.hflip(batch)    # Batch flippato orizzontalmente

        # Accumulator over the 8 views
        probs_sum = torch.zeros(batch.size(0), 4, device=device) # 4 classi

        # Le 4 rotazioni dell'originale
        probs_sum += torch.softmax(model(batch), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch, 90)), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch, 180)), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch, 270)), dim=1)

        # Le 4 rotazioni del flippato (copre tutte le combinazioni speculari)
        probs_sum += torch.softmax(model(batch_flip), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch_flip, 90)), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch_flip, 180)), dim=1)
        probs_sum += torch.softmax(model(TF.rotate(batch_flip, 270)), dim=1)

        # Average of the 8 predictions
        p = probs_sum / 8.0
        all_probs.append(p.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    return all_probs.mean(dim=0)

In [45]:
from collections import defaultdict

def build_slide_index(slide_ids):
    d = defaultdict(list)
    for i, sid in enumerate(slide_ids):
        d[sid].append(i)
    return d

train_slide_map = build_slide_index(train_slide_ids)
test_slide_map  = build_slide_index(test_slide_ids)

In [46]:
model.load_state_dict(torch.load("resnet18_multiclass_secondtry_1212.pth", map_location=device))

test_preds = []

for slide_id, idxs in test_slide_map.items():
    prob = predict_slide_tta_dihedral(
        model,
        test_tiles,
        idxs,
        val_tf
    )
    pred = prob.argmax().item()
    test_preds.append((slide_id, pred))

In [47]:
label_map = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}

submission = pd.DataFrame({
    "sample_index": [sid for sid, _ in test_preds],
    "label": [label_map[p] for _, p in test_preds]
})

submission.to_csv("submission_resnet18_multiclass_1212_ttadihedral.csv", index=False)
submission.head()

,sample_index,label
0,img_0000.png,Luminal A
1,img_0001.png,Luminal B
2,img_0002.png,Luminal B
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [48]:
import pandas as pd

CSV_OLD = "submission_resnet18_multiclass_1212_secondtry.csv"        # 0.3859
CSV_NEW = "submission_resnet18_multiclass_1212_ttadihedral.csv"  # step 2

# Load
df_old = pd.read_csv(CSV_OLD)
df_new = pd.read_csv(CSV_NEW)

# Safety checks
assert set(df_old.columns) == {"sample_index", "label"}
assert set(df_new.columns) == {"sample_index", "label"}

df_old = df_old.sort_values("sample_index").reset_index(drop=True)
df_new = df_new.sort_values("sample_index").reset_index(drop=True)

assert (df_old["sample_index"] == df_new["sample_index"]).all(), \
    "Mismatch in sample_index ordering"

# Diff
diff = df_old.copy()
diff["label_old"] = df_old["label"]
diff["label_new"] = df_new["label"]
diff["changed"] = diff["label_old"] != diff["label_new"]

n_total = len(diff)
n_changed = diff["changed"].sum()

print(f"Total samples: {n_total}")
print(f"Changed predictions: {n_changed} ({100*n_changed/n_total:.2f}%)")

# Show distribution of changes
if n_changed > 0:
    print("\nChange breakdown (old → new):")
    print(
        diff[diff["changed"]]
        .groupby(["label_old", "label_new"])
        .size()
        .sort_values(ascending=False)
    )

# Optional: show first N changed samples
N_SHOW = 20
if n_changed > 0:
    print(f"\nFirst {min(N_SHOW, n_changed)} changed samples:")
    display(
        diff[diff["changed"]]
        .head(N_SHOW)[["sample_index", "label_old", "label_new"]]
    )
else:
    print("\nNo differences found between submissions.")

Total samples: 477
Changed predictions: 37 (7.76%)

Change breakdown (old → new):
label_old        label_new
HER2(+)          Luminal B          5
Triple negative  Luminal B          5
Luminal B        Triple negative    5
                 Luminal A          5
Luminal A        HER2(+)            3
Luminal B        HER2(+)            3
HER2(+)          Triple negative    2
                 Luminal A          2
Triple negative  HER2(+)            2
Luminal A        Luminal B          2
Triple negative  Luminal A          2
Luminal A        Triple negative    1
dtype: int64

First 20 changed samples:


,sample_index,label_old,label_new
5,img_0005.png,Luminal B,Triple negative
13,img_0013.png,Luminal B,Triple negative
52,img_0052.png,Luminal B,Luminal A
61,img_0061.png,HER2(+),Luminal A
65,img_0065.png,Luminal A,HER2(+)
68,img_0068.png,HER2(+),Luminal B
110,img_0110.png,Luminal B,HER2(+)
122,img_0122.png,Luminal B,HER2(+)
128,img_0128.png,Luminal A,HER2(+)
152,img_0152.png,Triple negative,Luminal A
